# Layer 2: leave-one-farm-out evaluation

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "layer2_foldwise.py").exists():
        sys.path.insert(0, str(_cand))
        break
import layer2_foldwise as fw

In [ ]:
ROOT = fw.ROOT
OUT_DIR = ROOT / "output" / "14_layer2_crossfarm_eval"
BIAS_DIR = ROOT / "output" / "15_layer2_bias_sensitivity"
LOG_DIR = ROOT / "output" / "logs"
for _d in (OUT_DIR, BIAS_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOG_PATH = LOG_DIR / f"13_layer2_crossfarm_bias_eval_{RUN_TAG}.log"


def log(msg: str):
    line = f"[{datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S UTC}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


log("Notebook started.")

In [ ]:
corpus = fw.load_corpus()
seg_vectors = corpus["seg_vectors"]
seg_clip = corpus["seg_clip"]
seg_index = corpus["seg_index"]
clips = corpus["clips"]
farm_of = dict(zip(clips["clip_id"], clips["farm"]))
FARMS = [1, 2, 3]

TARGET_SPECS = [
    {"target_name": "proxy_functional_v1", "task_type": "multiclass"},
    {"target_name": "is_nonvocal", "task_type": "binary"},
    {"target_name": "is_distress_related", "task_type": "binary"},
    {"target_name": "is_resource_related", "task_type": "binary"},
]
labels = {s["target_name"]: dict(zip(clips["clip_id"], clips[s["target_name"]])) for s in TARGET_SPECS}


def make_model(task_type: str):
    if task_type == "multiclass":
        return LogisticRegression(max_iter=4000, solver="saga", class_weight="balanced", random_state=42)
    return LogisticRegression(max_iter=4000, solver="liblinear", class_weight="balanced", random_state=42)


def bundle_keys():
    return [(fam, vocab, ctx)
            for fam in fw.AUDIO_FAMILIES
            for vocab in fw.VOCAB_VARIANTS
            for ctx in fw.CONTEXT_VARIANTS]


def bundle_name(key):
    fam, vocab, ctx = key
    return f"{fam}_{vocab}__{ctx}"


def macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


log(f"Corpus: {len(clips)} farm-identified clips, {len(seg_vectors)} segments, "
    f"K={fw.K}, PCA={fw.PCA_DIM}.")

In [ ]:
pred_records = []
selection_records = []

for held in FARMS:
    seg_train_mask = np.array([farm_of[c] != held for c in seg_clip])
    assign = fw.fit_fold_vocabulary(seg_vectors, seg_train_mask)
    tokens = assign(seg_vectors)
    sequences = fw.build_token_sequences(tokens, seg_clip, seg_index)

    train_clips = [c for c in sequences if farm_of[c] != held]
    test_clips = [c for c in sequences if farm_of[c] == held]
    ordered = train_clips + test_clips
    retained_units = fw.audit_filtered_units(tokens, seg_clip, set(train_clips), clips)

    matrices = {}
    for key in bundle_keys():
        fam, vocab, ctx = key
        audio = fw.build_audio_matrix(fam, vocab, sequences, train_clips, ordered, retained_units)
        context = fw.build_context_matrix(ctx, clips, train_clips, ordered)
        matrices[key] = audio if context is None else np.hstack([audio, context])

    row_of = {c: i for i, c in enumerate(ordered)}
    train_rows = [row_of[c] for c in train_clips]
    test_rows = [row_of[c] for c in test_clips]
    inner_groups = np.array([farm_of[c] for c in train_clips])
    inner_cv = LeaveOneGroupOut()

    for spec in TARGET_SPECS:
        target = spec["target_name"]
        task = spec["task_type"]
        y_train_raw = np.array([labels[target][c] for c in train_clips])
        y_test_raw = np.array([labels[target][c] for c in test_clips])
        if task == "multiclass":
            encoder = LabelEncoder().fit(y_train_raw)
            y_train = encoder.transform(y_train_raw)
            y_test = encoder.transform(y_test_raw)
            class_labels = encoder.classes_.tolist()
        else:
            y_train = y_train_raw.astype(int)
            y_test = y_test_raw.astype(int)
            class_labels = [0, 1]

        best_key, best_score = None, -1.0
        for key in matrices:
            X_train = matrices[key][train_rows]
            inner_scores = []
            for itr, iva in inner_cv.split(X_train, y_train, inner_groups):
                model = make_model(task).fit(X_train[itr], y_train[itr])
                inner_scores.append(macro_f1(y_train[iva], model.predict(X_train[iva])))
            mean_score = float(np.mean(inner_scores))
            selection_records.append({
                "heldout_farm": str(held),
                "target_name": target,
                "bundle_name": bundle_name(key),
                "inner_macro_f1": mean_score,
            })
            if mean_score > best_score:
                best_score, best_key = mean_score, key

        X_train = matrices[best_key][train_rows]
        X_test = matrices[best_key][test_rows]
        model = make_model(task).fit(X_train, y_train)
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
        y_dummy = dummy.predict(X_test)

        for i, clip_id in enumerate(test_clips):
            rec = {
                "_record_key": clip_id,
                "target_name": target,
                "task_type": task,
                "model_name": "logreg_balanced",
                "heldout_farm": str(held),
                "bundle_name": bundle_name(best_key),
                "audio_feature_name": f"{best_key[0]}_{best_key[1]}",
                "variant": best_key[2],
                "y_true": int(y_test[i]),
                "y_pred": int(y_pred[i]),
                "y_dummy": int(y_dummy[i]),
            }
            if task == "multiclass":
                rec["y_true_label"] = class_labels[y_test[i]]
                rec["y_pred_label"] = class_labels[y_pred[i]]
                for j, name in enumerate(class_labels):
                    rec[f"proba__{name}"] = float(proba[i, j])
            else:
                rec["proba__1"] = float(proba[i, 1])
            pred_records.append(rec)

        log(f"farm={held} target={target} selected={bundle_name(best_key)} inner_macro_f1={best_score:.3f}")

pred_df = pd.DataFrame(pred_records)
selection_df = pd.DataFrame(selection_records)

In [ ]:
metric_rows = []
perfarm_rows = []

for spec in TARGET_SPECS:
    target = spec["target_name"]
    task = spec["task_type"]
    sub = pred_df[pred_df["target_name"] == target]
    y_true = sub["y_true"].to_numpy()
    y_pred = sub["y_pred"].to_numpy()
    y_dummy = sub["y_dummy"].to_numpy()

    for model_name, preds in (("logreg_balanced", y_pred), ("dummy_most_frequent", y_dummy)):
        metric_rows.append({
            "target_name": target,
            "task_type": task,
            "model_name": model_name,
            "n_samples": int(len(sub)),
            "macro_f1": macro_f1(y_true, preds),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
            "accuracy": float(accuracy_score(y_true, preds)),
            "macro_precision": float(precision_score(y_true, preds, average="macro", zero_division=0)),
            "macro_recall": float(recall_score(y_true, preds, average="macro", zero_division=0)),
        })

    for farm, pf in sub.groupby("heldout_farm"):
        perfarm_rows.append({
            "target_name": target,
            "heldout_farm": farm,
            "n_samples": int(len(pf)),
            "macro_f1": macro_f1(pf["y_true"], pf["y_pred"]),
            "balanced_accuracy": float(balanced_accuracy_score(pf["y_true"], pf["y_pred"])),
        })

metrics_df = pd.DataFrame(metric_rows)
perfarm_df = pd.DataFrame(perfarm_rows)

In [ ]:
metrics_path = OUT_DIR / "crossfarm_metrics_summary.csv"
preds_path = OUT_DIR / "crossfarm_fold_predictions.csv.gz"
selection_path = OUT_DIR / "crossfarm_bundle_selection_scores.csv"
perfarm_path = OUT_DIR / "crossfarm_bestbundle_perfarm_metrics.csv"
manifest_path = OUT_DIR / "crossfarm_manifest.json"

metrics_df.to_csv(metrics_path, index=False)
pred_df.to_csv(preds_path, index=False, compression="gzip")
selection_df.to_csv(selection_path, index=False)
perfarm_df.to_csv(perfarm_path, index=False)

manifest = {
    "build_ts_utc": RUN_TS,
    "n_clips": int(len(clips)),
    "farms": FARMS,
    "n_segments": int(len(seg_vectors)),
    "vocabulary_size": fw.K,
    "pca_components": fw.PCA_DIM,
    "cluster_seed": fw.CLUSTER_SEED,
    "targets": [s["target_name"] for s in TARGET_SPECS],
    "audio_families": fw.AUDIO_FAMILIES,
    "vocab_variants": fw.VOCAB_VARIANTS,
    "context_variants": fw.CONTEXT_VARIANTS,
    "outputs": {
        "metrics_summary": metrics_path.name,
        "fold_predictions": preds_path.name,
        "bundle_selection_scores": selection_path.name,
        "perfarm_metrics": perfarm_path.name,
    },
}
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

log("Saved cross-farm evaluation outputs.")

In [ ]:
display(metrics_df[metrics_df["model_name"] == "logreg_balanced"][
    ["target_name", "macro_f1", "balanced_accuracy", "accuracy", "n_samples"]
].reset_index(drop=True))
display(perfarm_df)